# SwingRL Weekly Performance Review

**Review Date:** _Run this notebook weekly to analyze trading performance._

Covers: portfolio curves, trade logs, risk metrics, feature pipeline health, system health, and shadow mode status.

In [ ]:
"""Setup: imports, DB paths, connections."""

from __future__ import annotations

import sqlite3
from datetime import UTC, datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Configure matplotlib
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

# DB paths (adjust if running outside repo root)
SQLITE_PATH = Path("db/trading_ops.db")
DUCKDB_PATH = Path("data/db/market_data.ddb")

# Time window
now_utc = datetime.now(UTC)
seven_days_ago = now_utc - timedelta(days=7)
date_filter = seven_days_ago.strftime("%Y-%m-%d")

print(f"Review period: {date_filter} to {now_utc.strftime('%Y-%m-%d')}")
print(f"SQLite: {SQLITE_PATH} (exists: {SQLITE_PATH.exists()})")
print(f"DuckDB: {DUCKDB_PATH} (exists: {DUCKDB_PATH.exists()})")

## Portfolio Performance

In [ ]:
"""Query portfolio_snapshots for last 7 days per environment."""

if SQLITE_PATH.exists():
    conn = sqlite3.connect(str(SQLITE_PATH))
    query = f"""
        SELECT timestamp, environment, total_value, cash_balance
        FROM portfolio_snapshots
        WHERE timestamp >= '{date_filter}'
        ORDER BY timestamp
    """
    df_portfolio = pd.read_sql_query(query, conn)
    conn.close()

    if not df_portfolio.empty:
        df_portfolio["timestamp"] = pd.to_datetime(df_portfolio["timestamp"])

        fig, ax = plt.subplots(figsize=(12, 6))
        for env in df_portfolio["environment"].unique():
            env_data = df_portfolio[df_portfolio["environment"] == env]
            ax.plot(env_data["timestamp"], env_data["total_value"], label=env, linewidth=2)

        ax.set_title("Portfolio Value (Last 7 Days)")
        ax.set_xlabel("Date")
        ax.set_ylabel("Total Value (USD)")
        ax.legend()
        plt.tight_layout()
        plt.show()

        # Summary stats
        latest = df_portfolio.groupby("environment").last()
        earliest = df_portfolio.groupby("environment").first()
        print("\nCurrent Portfolio Values:")
        for env in latest.index:
            current = latest.loc[env, "total_value"]
            prev = earliest.loc[env, "total_value"]
            change_pct = ((current - prev) / prev * 100) if prev else 0
            print(f"  {env}: ${current:,.2f} (week: {change_pct:+.2f}%)")
    else:
        print("No portfolio snapshots in the last 7 days.")
else:
    print(f"SQLite DB not found at {SQLITE_PATH}")

## Trade Log

In [ ]:
"""Query trades table for last 7 days."""

if SQLITE_PATH.exists():
    conn = sqlite3.connect(str(SQLITE_PATH))
    query = f"""
        SELECT timestamp, environment, symbol, side, quantity, price, pnl
        FROM trades
        WHERE timestamp >= '{date_filter}'
        ORDER BY timestamp DESC
    """
    df_trades = pd.read_sql_query(query, conn)
    conn.close()

    if not df_trades.empty:
        print(f"Total trades this week: {len(df_trades)}")
        print("\nTrades per environment:")
        print(df_trades["environment"].value_counts().to_string())
        print("\nTrades per side:")
        print(df_trades["side"].value_counts().to_string())
        print("\n--- Recent Trades ---")
        display(df_trades.head(20))  # noqa: F821

        # Trade count bar chart
        fig, ax = plt.subplots(figsize=(8, 4))
        trade_counts = df_trades.groupby(["environment", "side"]).size().unstack(fill_value=0)
        trade_counts.plot(kind="bar", ax=ax)
        ax.set_title("Trade Counts by Environment and Side")
        ax.set_ylabel("Count")
        plt.tight_layout()
        plt.show()
    else:
        print("No trades in the last 7 days.")
else:
    print(f"SQLite DB not found at {SQLITE_PATH}")

## Risk Metrics

In [ ]:
"""Compute and display risk metrics per environment."""

if SQLITE_PATH.exists():
    conn = sqlite3.connect(str(SQLITE_PATH))

    # Current drawdown per environment
    query = f"""
        SELECT environment, total_value, timestamp
        FROM portfolio_snapshots
        WHERE timestamp >= '{date_filter}'
        ORDER BY timestamp
    """
    df_risk = pd.read_sql_query(query, conn)

    if not df_risk.empty:
        print("Risk Metrics (Last 7 Days)")
        print("=" * 40)
        for env in df_risk["environment"].unique():
            env_data = df_risk[df_risk["environment"] == env]
            peak = env_data["total_value"].cummax()
            drawdown = (env_data["total_value"] - peak) / peak
            current_dd = drawdown.iloc[-1]
            max_dd = drawdown.min()
            print(f"\n  {env}:")
            print(f"    Current drawdown: {current_dd:.2%}")
            print(f"    Max drawdown (week): {max_dd:.2%}")

    # Circuit breaker events
    cb_query = f"""
        SELECT timestamp, environment, event_type, detail
        FROM circuit_breaker_events
        WHERE timestamp >= '{date_filter}'
        ORDER BY timestamp DESC
    """
    try:
        df_cb = pd.read_sql_query(cb_query, conn)
        if not df_cb.empty:
            print(f"\nCircuit Breaker Events: {len(df_cb)}")
            display(df_cb)  # noqa: F821
        else:
            print("\nNo circuit breaker events this week.")
    except Exception:
        print("\nCircuit breaker events table not yet created.")

    conn.close()
else:
    print(f"SQLite DB not found at {SQLITE_PATH}")

## Feature Pipeline Health

In [ ]:
"""Query data_ingestion_log for last 7 days."""

if SQLITE_PATH.exists():
    conn = sqlite3.connect(str(SQLITE_PATH))
    query = f"""
        SELECT timestamp, source, status, rows_fetched, detail
        FROM data_ingestion_log
        WHERE timestamp >= '{date_filter}'
        ORDER BY timestamp DESC
    """
    try:
        df_ingest = pd.read_sql_query(query, conn)
        if not df_ingest.empty:
            total = len(df_ingest)
            success = len(df_ingest[df_ingest["status"] == "success"])
            print(f"Ingestion runs: {total}, Success: {success} ({success / total:.0%})")
            print("\nBy source:")
            print(df_ingest.groupby(["source", "status"]).size().unstack(fill_value=0).to_string())

            # Check for quarantine events
            quarantined = df_ingest[df_ingest["status"] == "quarantined"]
            if not quarantined.empty:
                print(f"\nQuarantine events: {len(quarantined)}")
                display(quarantined)  # noqa: F821
        else:
            print("No ingestion log entries in the last 7 days.")
    except Exception:
        print("Data ingestion log table not yet created.")
    conn.close()
else:
    print(f"SQLite DB not found at {SQLITE_PATH}")

## System Health

In [ ]:
"""Show backup status, scheduler execution, and error log entries."""

# Recent backups
backup_dir = Path("backups")
print("Recent Backups:")
print("=" * 40)
for subdir in ["sqlite", "duckdb"]:
    bdir = backup_dir / subdir
    if bdir.exists():
        files = sorted(bdir.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)[:5]
        print(f"\n  {subdir}:")
        for f in files:
            print(f"    {f.name} ({f.stat().st_size / 1024:.1f} KB)")
    else:
        print(f"\n  {subdir}: directory not found")

# Scheduler job execution
if SQLITE_PATH.exists():
    conn = sqlite3.connect(str(SQLITE_PATH))
    try:
        query = f"""
            SELECT job_name, COUNT(*) as runs,
                   SUM(CASE WHEN status='success' THEN 1 ELSE 0 END) as success,
                   MAX(timestamp) as last_run
            FROM scheduler_log
            WHERE timestamp >= '{date_filter}'
            GROUP BY job_name
        """
        df_sched = pd.read_sql_query(query, conn)
        if not df_sched.empty:
            print("\nScheduler Jobs (Last 7 Days):")
            display(df_sched)  # noqa: F821
        else:
            print("\nNo scheduler log entries.")
    except Exception:
        print("\nScheduler log table not yet created.")
    conn.close()

# Error log entries
log_dir = Path("logs")
if log_dir.exists():
    log_files = sorted(log_dir.glob("*.log"), key=lambda p: p.stat().st_mtime, reverse=True)[:3]
    print(f"\nRecent Log Files: {[f.name for f in log_files]}")
else:
    print("\nLogs directory not found.")

## Shadow Mode Status

In [ ]:
"""Show shadow model performance vs active, promotion status."""

shadow_dir = Path("models/shadow")
if shadow_dir.exists() and any(shadow_dir.iterdir()):
    print("Shadow Models:")
    for model_file in shadow_dir.glob("*.zip"):
        print(f"  {model_file.name}")

    # Query shadow trades
    if SQLITE_PATH.exists():
        conn = sqlite3.connect(str(SQLITE_PATH))
        try:
            query = f"""
                SELECT environment, COUNT(*) as trade_count,
                       AVG(pnl) as avg_pnl,
                       SUM(pnl) as total_pnl
                FROM shadow_trades
                WHERE timestamp >= '{date_filter}'
                GROUP BY environment
            """
            df_shadow = pd.read_sql_query(query, conn)
            if not df_shadow.empty:
                print("\nShadow Trade Performance (Last 7 Days):")
                display(df_shadow)  # noqa: F821

                # Compare with active
                active_query = f"""
                    SELECT environment, COUNT(*) as trade_count,
                           AVG(pnl) as avg_pnl,
                           SUM(pnl) as total_pnl
                    FROM trades
                    WHERE timestamp >= '{date_filter}'
                    GROUP BY environment
                """
                df_active = pd.read_sql_query(active_query, conn)
                if not df_active.empty:
                    print("\nActive Trade Performance (Last 7 Days):")
                    display(df_active)  # noqa: F821
            else:
                print("\nNo shadow trades in the last 7 days.")
        except Exception:
            print("\nShadow trades table not yet created.")
        conn.close()

    # Promotion status
    if SQLITE_PATH.exists():
        conn = sqlite3.connect(str(SQLITE_PATH))
        try:
            query = """
                SELECT * FROM model_promotions
                ORDER BY timestamp DESC
                LIMIT 5
            """
            df_promo = pd.read_sql_query(query, conn)
            if not df_promo.empty:
                print("\nRecent Model Promotions:")
                display(df_promo)  # noqa: F821
            else:
                print("\nNo model promotions yet.")
        except Exception:
            print("\nModel promotions table not yet created.")
        conn.close()
else:
    print("No shadow models deployed.")